In [5]:
# ============================================================================
# BALANCE HÍDRICO SUBTERRÁNEO CORREGIDO - PARTE 1
# Ecuaciones basadas en literatura hidrológica
# PASOS 1-5: Carga de datos, sincronización y cálculo de ecuaciones
# ============================================================================

import pandas as pd
import numpy as np
from geopy.distance import geodesic
import warnings
import os
import traceback

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURACIÓN INICIAL
# ============================================================================

print("\n" + "="*150)
print("BALANCE HÍDRICO SUBTERRÁNEO CORREGIDO - PARTE 1 (PASOS 1-5)")
print("="*150)

INPUT_DIR = r'C:\Users\omari\Documents\12 semestre\tesis\analisis multiscala\SMN_data_puebla_operativas\processed'
OUTPUT_DIR = r'C:\Users\omari\Documents\12 semestre\tesis\analisis multiscala\SMN_data_puebla_operativas\water_balance_analysis'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Datos CONAGUA por acuífero
datos_conagua = {
    '2101': {
        'nombre': 'Valle de Tecamachalco',
        'area_km2': 3340,
        'VEAS': 217.756614,
        'DNC': 0.0,
    },
    '2102': {
        'nombre': 'Libres-Oriental',
        'area_km2': 3995,
        'VEAS': 159.771638,
        'DNC': 0.0,
    },
    '2103': {
        'nombre': 'Atlixco-Izúcar de Matamoros',
        'area_km2': 2658,
        'VEAS': 123.904179,
        'DNC': 83.9,
    },
    '2104': {
        'nombre': 'Valle de Puebla',
        'area_km2': 2025,
        'VEAS': 281.482546,
        'DNC': 61.2,
    },
    '2106': {
        'nombre': 'Ixcaquixtla',
        'area_km2': 9559,
        'VEAS': 74.929168,
        'DNC': 1.3,
    }
}

area_total_km2 = sum(d['area_km2'] for d in datos_conagua.values())
balance_por_acuifero = {codigo: {} for codigo in datos_conagua.keys()}

# ============================================================================
# PASO 1: CARGAR DATOS DE PRECIPITACIÓN Y EVAPORACIÓN
# ============================================================================

print("\n[PASO 1] Cargando datos de precipitación y evaporación...")
print("-" * 150)

try:
    precip_df = pd.read_csv(f'{INPUT_DIR}/all_stations_precipitation.csv', 
                            index_col=0, parse_dates=True)
    print(f"✓ Precipitación SMN cargada: {len(precip_df)} registros, {len(precip_df.columns)} estaciones")
    
    evap_df = pd.read_csv(f'{INPUT_DIR}/all_stations_evaporation.csv', 
                          index_col=0, parse_dates=True)
    print(f"✓ Evaporación SMN cargada: {len(evap_df)} registros, {len(evap_df.columns)} estaciones")
    
    if len(precip_df) != len(evap_df):
        print(f"⚠ Advertencia: diferentes longitudes (Precip: {len(precip_df)}, Evap: {len(evap_df)})")
        common_index = precip_df.index.intersection(evap_df.index)
        precip_df = precip_df.loc[common_index]
        evap_df = evap_df.loc[common_index]
        print(f"  → Sincronizadas a período común: {len(precip_df)} registros")
        
except Exception as e:
    print(f"✗ Error cargando datos SMN: {e}")
    traceback.print_exc()
    exit(1)

# ============================================================================
# PASO 2: CARGAR DATOS DE INFILTRACIÓN (GRACE)
# ============================================================================

print("\n[PASO 2] Cargando datos de infiltración (GRACE)...")
print("-" * 150)

try:
    grace_combinado = pd.read_csv(f'{INPUT_DIR}/grace_smn_combined.csv', 
                                  index_col=0, parse_dates=True)
    grace_anual = pd.read_csv(f'{INPUT_DIR}/grace_annual_summary.csv', 
                              index_col=0)
    
    print(f"✓ GRACE mensual: {len(grace_combinado)} registros")
    print(f"✓ GRACE anual: {len(grace_anual)} años")
    print(f"  Columnas GRACE: {list(grace_combinado.columns)}")
    
except Exception as e:
    print(f"✗ Error cargando GRACE: {e}")
    traceback.print_exc()
    exit(1)

# ============================================================================
# PASO 2.5: SINCRONIZAR DATOS DE PRECIPITACIÓN Y EVAPORACIÓN
# ============================================================================

print("\n[PASO 2.5] Sincronizando datos de precipitación y evaporación...")
print("-" * 150)

print(f"\n  Antes de sincronizar:")
print(f"    Precipitación columnas: {len(precip_df.columns)}")
print(f"    Evaporación columnas: {len(evap_df.columns)}")

columnas_validas = precip_df.columns.intersection(evap_df.columns)
precip_df = precip_df[columnas_validas]
evap_df = evap_df[columnas_validas]

print(f"\n  Después de sincronizar:")
print(f"    Precipitación columnas: {len(precip_df.columns)}")
print(f"    Evaporación columnas: {len(evap_df.columns)}")
print(f"    ✓ Ambos DataFrames ahora tienen {len(columnas_validas)} estaciones comunes")

# ============================================================================
# PASO 3: CARGAR COORDENADAS Y ASIGNAR ESTACIONES A ACUÍFEROS
# ============================================================================

print("\n[PASO 3] Asignando estaciones a acuíferos...")
print("-" * 150)

stations_coords = None
coord_file = f'{INPUT_DIR}/stations_coordinates.csv'

print(f"\n📂 Buscando archivo: {coord_file}")

if not os.path.exists(coord_file):
    print(f"✗ Archivo no encontrado: {coord_file}")
    print(f"  Archivos disponibles en {INPUT_DIR}:")
    try:
        archivos = os.listdir(INPUT_DIR)
        for archivo in archivos:
            print(f"    - {archivo}")
    except:
        pass
else:
    print(f"✓ Archivo encontrado!")
    
    try:
        print(f"\n🔍 Leyendo archivo...")
        stations_coords_raw = pd.read_csv(coord_file)
        
        print(f"✓ Archivo cargado")
        print(f"  Shape: {stations_coords_raw.shape}")
        print(f"  Columnas: {list(stations_coords_raw.columns)}")
        print(f"  Primeras 3 filas:")
        print(stations_coords_raw.head(3).to_string())
        
        print(f"\n🧹 Validando datos...")
        stations_coords_raw.columns = [col.lower().strip() for col in stations_coords_raw.columns]
        print(f"  Columnas normalizadas: {list(stations_coords_raw.columns)}")
        
        lon_cols = [col for col in stations_coords_raw.columns if 'lon' in col.lower()]
        lat_cols = [col for col in stations_coords_raw.columns if 'lat' in col.lower()]
        alt_cols = [col for col in stations_coords_raw.columns if 'alt' in col.lower()]
        station_id_cols = [col for col in stations_coords_raw.columns if 'station' in col.lower() or 'id' in col.lower()]
        
        print(f"\n  ✓ Columnas identificadas:")
        print(f"    - Longitud: {lon_cols}")
        print(f"    - Latitud: {lat_cols}")
        print(f"    - Altitud: {alt_cols}")
        print(f"    - Station ID: {station_id_cols}")
        
        if not lon_cols or not lat_cols:
            print(f"  ✗ No se encontraron columnas de coordenadas")
            print(f"    Intentando con índices por posición...")
            stations_coords = pd.DataFrame()
            stations_coords['longitude'] = pd.to_numeric(stations_coords_raw.iloc[:, 0], errors='coerce')
            stations_coords['latitude'] = pd.to_numeric(stations_coords_raw.iloc[:, 1], errors='coerce')
            stations_coords['altitude'] = pd.to_numeric(stations_coords_raw.iloc[:, 2], errors='coerce')
            stations_coords['station_id'] = stations_coords_raw.iloc[:, 3].astype(str)
        else:
            stations_coords = pd.DataFrame()
            stations_coords['longitude'] = pd.to_numeric(stations_coords_raw[lon_cols[0]], errors='coerce')
            stations_coords['latitude'] = pd.to_numeric(stations_coords_raw[lat_cols[0]], errors='coerce')
            
            if alt_cols:
                stations_coords['altitude'] = pd.to_numeric(stations_coords_raw[alt_cols[0]], errors='coerce')
            else:
                stations_coords['altitude'] = np.nan
            
            if station_id_cols:
                stations_coords['station_id'] = stations_coords_raw[station_id_cols[0]].astype(str)
            else:
                stations_coords['station_id'] = range(len(stations_coords_raw))
        
        print(f"\n  📊 Estadísticas antes de limpieza:")
        print(f"    Total filas: {len(stations_coords)}")
        print(f"    Longitud - Nulls: {stations_coords['longitude'].isnull().sum()}")
        print(f"    Latitud - Nulls: {stations_coords['latitude'].isnull().sum()}")
        
        stations_coords_limpio = stations_coords.dropna(subset=['longitude', 'latitude'])
        
        print(f"\n  📊 Estadísticas después de limpieza:")
        print(f"    Total filas: {len(stations_coords_limpio)}")
        print(f"    Filas eliminadas: {len(stations_coords) - len(stations_coords_limpio)}")
        
        print(f"\n  🌍 Validando rangos geográficos:")
        print(f"    Latitud range: {stations_coords_limpio['latitude'].min():.6f} to {stations_coords_limpio['latitude'].max():.6f}")
        print(f"    Longitud range: {stations_coords_limpio['longitude'].min():.6f} to {stations_coords_limpio['longitude'].max():.6f}")
        
        lat_valida = (stations_coords_limpio['latitude'] >= 14) & (stations_coords_limpio['latitude'] <= 33)
        lon_valida = (stations_coords_limpio['longitude'] >= -117) & (stations_coords_limpio['longitude'] <= -86)
        estaciones_validas = lat_valida & lon_valida
        
        print(f"    Estaciones válidas: {estaciones_validas.sum()}/{len(stations_coords_limpio)}")
        
        if estaciones_validas.sum() < len(stations_coords_limpio):
            print(f"    ⚠ {(~estaciones_validas).sum()} estaciones fuera de rango - eliminando...")
            stations_coords_limpio = stations_coords_limpio[estaciones_validas]
        
        stations_coords = stations_coords_limpio
        
        if len(stations_coords) > 0:
            print(f"\n  ✅ COORDENADAS CARGADAS EXITOSAMENTE!")
            print(f"     Total estaciones válidas: {len(stations_coords)}")
        else:
            print(f"\n  ❌ No hay coordenadas válidas después de limpieza")
            stations_coords = None
        
    except Exception as e:
        print(f"✗ Error leyendo archivo: {e}")
        traceback.print_exc()
        stations_coords = None

# ============================================================================
# PASO 3.4: ASIGNACIÓN GEOGRÁFICA DE ESTACIONES A ACUÍFEROS
# ============================================================================

print("\n" + "-" * 150)
print("[PASO 3.4] Asignando estaciones a acuíferos por proximidad geográfica...")
print("-" * 150)

aquifer_info = {
    '2101': {
        'nombre': 'Valle de Tecamachalco',
        'centroid': (18.7, -97.7),
        'bounds': {'lat_min': 18.5, 'lat_max': 19.0, 'lon_min': -98.0, 'lon_max': -97.4}
    },
    '2102': {
        'nombre': 'Libres-Oriental',
        'centroid': (19.4, -97.6),
        'bounds': {'lat_min': 19.2, 'lat_max': 19.6, 'lon_min': -98.0, 'lon_max': -97.2}
    },
    '2103': {
        'nombre': 'Atlixco-Izúcar de Matamoros',
        'centroid': (18.6, -98.4),
        'bounds': {'lat_min': 18.3, 'lat_max': 18.9, 'lon_min': -98.7, 'lon_max': -98.1}
    },
    '2104': {
        'nombre': 'Valle de Puebla',
        'centroid': (19.0, -98.2),
        'bounds': {'lat_min': 18.8, 'lat_max': 19.2, 'lon_min': -98.5, 'lon_max': -97.9}
    },
    '2106': {
        'nombre': 'Ixcaquixtla',
        'centroid': (18.2, -97.9),
        'bounds': {'lat_min': 17.9, 'lat_max': 18.5, 'lon_min': -98.3, 'lon_max': -97.5}
    }
}

def assign_station_to_aquifer(lat, lon, aquifer_info, max_distance_km=75):
    """Asigna estación a acuífero usando bounding box y proximidad"""
    for aquifer_code, info in aquifer_info.items():
        bounds = info['bounds']
        if (bounds['lat_min'] <= lat <= bounds['lat_max'] and 
            bounds['lon_min'] <= lon <= bounds['lon_max']):
            return aquifer_code
    
    min_distance = float('inf')
    nearest_aquifer = None
    
    for aquifer_code, info in aquifer_info.items():
        centroid = info['centroid']
        distance = geodesic((lat, lon), centroid).kilometers
        
        if distance < min_distance and distance <= max_distance_km:
            min_distance = distance
            nearest_aquifer = aquifer_code
    
    return nearest_aquifer

if stations_coords is not None and len(stations_coords) > 0:
    
    station_assignments = {codigo: [] for codigo in datos_conagua.keys()}
    station_details = []
    unassigned_stations = []
    
    print(f"\n  Asignando {len(stations_coords)} estaciones...")
    
    for idx, (_, station) in enumerate(stations_coords.iterrows()):
        station_id = station['station_id']
        lat = station['latitude']
        lon = station['longitude']
        
        assigned_aquifer = assign_station_to_aquifer(lat, lon, aquifer_info)
        
        if assigned_aquifer:
            station_assignments[assigned_aquifer].append(station_id)
            station_details.append({
                'station_id': station_id,
                'latitude': lat,
                'longitude': lon,
                'aquifer_code': assigned_aquifer,
                'aquifer_name': datos_conagua[assigned_aquifer]['nombre']
            })
        else:
            unassigned_stations.append(station_id)
    
    print(f"\n  📊 Resumen de asignación:")
    print(f"     Estaciones asignadas: {sum(len(v) for v in station_assignments.values())}")
    print(f"     Estaciones sin asignar: {len(unassigned_stations)}")
    
    print(f"\n  🔍 Verificando disponibilidad de datos:")
    print(f"     Columnas en precip_df: {len(precip_df.columns)}")
    print(f"     Columnas en evap_df: {len(evap_df.columns)}")
    
    columnas_comunes = precip_df.columns.intersection(evap_df.columns)
    print(f"     Columnas comunes: {len(columnas_comunes)}")
    
    print(f"\n  📊 Procesando datos por acuífero:")
    
    for codigo_acuifero in sorted(datos_conagua.keys()):
        station_list = station_assignments[codigo_acuifero]
        info_acuifero = datos_conagua[codigo_acuifero]
        
        if station_list:
            station_list_str = [str(st) for st in station_list]
            cols_en_precip = [st for st in station_list_str if st in precip_df.columns]
            cols_comunes = [st for st in cols_en_precip if st in evap_df.columns]
            
            print(f"\n     {codigo_acuifero}: {info_acuifero['nombre']}")
            print(f"        Asignadas: {len(station_list)} | En precip: {len(cols_en_precip)} | En ambas: {len(cols_comunes)}")
            
            if len(cols_comunes) > 0:
                try:
                    precip_acuifero = precip_df[cols_comunes].mean(axis=1)
                    evap_acuifero = evap_df[cols_comunes].mean(axis=1)
                    
                    balance_por_acuifero[codigo_acuifero]['precipitacion_mm'] = precip_acuifero
                    balance_por_acuifero[codigo_acuifero]['evapotranspiracion_mm'] = evap_acuifero
                    
                    mean_annual_precip = precip_acuifero.resample('Y').sum().mean()
                    mean_annual_evap = evap_acuifero.resample('Y').sum().mean()
                    
                    print(f"        ✓ Usando {len(cols_comunes)} estaciones")
                    print(f"          P: {mean_annual_precip:>6.1f} mm/yr | ET: {mean_annual_evap:>6.1f} mm/yr")
                    
                except Exception as e:
                    print(f"        ✗ Error: {e} - USANDO PROMEDIO REGIONAL")
                    precip_acuifero = precip_df.mean(axis=1)
                    evap_acuifero = evap_df.mean(axis=1)
                    balance_por_acuifero[codigo_acuifero]['precipitacion_mm'] = precip_acuifero
                    balance_por_acuifero[codigo_acuifero]['evapotranspiracion_mm'] = evap_acuifero
            else:
                print(f"        ⚠ USANDO PROMEDIO REGIONAL")
                precip_acuifero = precip_df.mean(axis=1)
                evap_acuifero = evap_df.mean(axis=1)
                balance_por_acuifero[codigo_acuifero]['precipitacion_mm'] = precip_acuifero
                balance_por_acuifero[codigo_acuifero]['evapotranspiracion_mm'] = evap_acuifero
        else:
            print(f"\n     {codigo_acuifero}: {info_acuifero['nombre']}")
            print(f"        ✗ Sin estaciones - USANDO PROMEDIO REGIONAL")
            precip_acuifero = precip_df.mean(axis=1)
            evap_acuifero = evap_df.mean(axis=1)
            balance_por_acuifero[codigo_acuifero]['precipitacion_mm'] = precip_acuifero
            balance_por_acuifero[codigo_acuifero]['evapotranspiracion_mm'] = evap_acuifero
    
    if station_details:
        df_asignaciones = pd.DataFrame(station_details)
        df_asignaciones.to_csv(f'{OUTPUT_DIR}/stations_aquifer_assignments.csv', index=False)
        print(f"\n  ✓ Guardado: stations_aquifer_assignments.csv")
        print(f"     {len(df_asignaciones)} estaciones asignadas")

else:
    print(f"\n  ⚠ Coordenadas no disponibles - usando promedio regional")
    for codigo_acuifero in datos_conagua.keys():
        balance_por_acuifero[codigo_acuifero]['precipitacion_mm'] = precip_df.mean(axis=1)
        balance_por_acuifero[codigo_acuifero]['evapotranspiracion_mm'] = evap_df.mean(axis=1)

print(f"\n✅ PASO 3 COMPLETADO")

# ============================================================================
# PASO 4: ECUACIÓN 1 - CALCULAR RECARGA DE AGUA (WATER RECHARGE)
# ============================================================================

print("\n[PASO 4] Calculando RECARGA DE AGUA por ecuación de balance hídrico...")
print("-" * 150)

print("\n📐 ECUACIÓN APLICADA:")
print("   Water Recharge = Precipitation - Evaporation + Infiltration")
print("   WR = P(SMN) - ET(SMN) + GW_Infiltration(GRACE)")
print("-" * 150)

for codigo_acuifero, info_acuifero in datos_conagua.items():
    
    fraccion_area = info_acuifero['area_km2'] / area_total_km2
    precip_media = balance_por_acuifero[codigo_acuifero]['precipitacion_mm']
    evap_media = balance_por_acuifero[codigo_acuifero]['evapotranspiracion_mm']
    
    grace_col = 'grace_tws_change_cm' if 'grace_tws_change_cm' in grace_combinado.columns else grace_combinado.columns[0]
    grace_cambio_cm = grace_combinado[grace_col]
    grace_cambio_mm = grace_cambio_cm * 10
    infiltracion_mm = grace_cambio_mm * fraccion_area
    
    balance_por_acuifero[codigo_acuifero]['infiltracion_mm'] = infiltracion_mm
    
    recarga_mm = precip_media - evap_media + infiltracion_mm
    balance_por_acuifero[codigo_acuifero]['recarga_mm'] = recarga_mm
    
    area_km2 = info_acuifero['area_km2']
    recarga_hm3 = recarga_mm * area_km2 * 0.001
    balance_por_acuifero[codigo_acuifero]['recarga_hm3'] = recarga_hm3
    
    precip_anual = precip_media.resample('Y').sum()
    evap_anual = evap_media.resample('Y').sum()
    infiltracion_anual = infiltracion_mm.resample('Y').sum()
    recarga_anual = recarga_mm.resample('Y').sum()
    recarga_anual_hm3 = recarga_hm3.resample('Y').sum()
    
    balance_por_acuifero[codigo_acuifero]['precip_anual'] = precip_anual
    balance_por_acuifero[codigo_acuifero]['evap_anual'] = evap_anual
    balance_por_acuifero[codigo_acuifero]['infiltracion_anual'] = infiltracion_anual
    balance_por_acuifero[codigo_acuifero]['recarga_anual_hm3'] = recarga_anual_hm3
    
    recarga_media_anual = recarga_anual_hm3.mean()
    recarga_std_anual = recarga_anual_hm3.std()
    
    print(f"\n{codigo_acuifero}: {info_acuifero['nombre']}")
    print(f"  Area: {area_km2:,} km² ({fraccion_area*100:.1f}% cuenca)")
    print(f"  ├─ Precipitación:        {precip_anual.mean():>8.1f} mm/año → {precip_anual.mean() * area_km2 * 0.001:>8.2f} hm³/año")
    print(f"  ├─ Evapotranspiración:   {evap_anual.mean():>8.1f} mm/año → {evap_anual.mean() * area_km2 * 0.001:>8.2f} hm³/año")
    print(f"  ├─ Infiltración (GRACE): {infiltracion_anual.mean():>8.1f} mm/año → {infiltracion_anual.mean() * area_km2 * 0.001:>8.2f} hm³/año")
    print(f"  └─ 💧 RECARGA TOTAL:      {recarga_media_anual:>8.2f} ± {recarga_std_anual:.2f} hm³/año")

# ============================================================================
# PASO 5: ECUACIÓN 2 - CALCULAR AGUA DISPONIBLE (WATER AVAILABLE)
# ============================================================================

print("\n\n[PASO 5] Calculando AGUA DISPONIBLE por ecuación de balance...")
print("-" * 150)

print("\n📐 ECUACIÓN APLICADA:")
print("   Water Available = Water Recharge - Extraction")
print("   WA = WR - VEAS(CONAGUA)")
print("-" * 150)

resultados_finales = {}

for codigo_acuifero, info_acuifero in datos_conagua.items():
    resultados = {}
    
    recarga_anual_hm3 = balance_por_acuifero[codigo_acuifero]['recarga_anual_hm3']
    recarga_promedio = recarga_anual_hm3.mean()
    recarga_std = recarga_anual_hm3.std()
    
    extraccion_anual = info_acuifero['VEAS']
    
    agua_disponible_anual = recarga_anual_hm3 - extraccion_anual
    agua_disponible_promedio = agua_disponible_anual.mean()
    agua_disponible_std = agua_disponible_anual.std()
    
    tasa_extraccion = (extraccion_anual / recarga_promedio * 100) if recarga_promedio > 0 else float('nan')
    
    resultados['recarga_promedio_hm3'] = recarga_promedio
    resultados['recarga_std_hm3'] = recarga_std
    resultados['extraccion_hm3'] = extraccion_anual
    resultados['agua_disponible_promedio_hm3'] = agua_disponible_promedio
    resultados['agua_disponible_std_hm3'] = agua_disponible_std
    resultados['tasa_extraccion_pct'] = tasa_extraccion
    
    resultados_finales[codigo_acuifero] = resultados
    
    if agua_disponible_promedio < 0:
        estado = "🔴 SOBREEEXPLOTADO"
    elif tasa_extraccion > 100:
        estado = "🔴 EN RIESGO CRÍTICO"
    elif tasa_extraccion > 80:
        estado = "🟡 EN RIESGO"
    else:
        estado = "🟢 SOSTENIBLE"
    
    print(f"\n{codigo_acuifero}: {info_acuifero['nombre']}")
    print(f"  ┌─ RECARGA:        {recarga_promedio:>10.2f} ± {recarga_std:>6.2f} hm³/año")
    print(f"  ├─ EXTRACCIÓN:    {extraccion_anual:>10.2f}           hm³/año")
    print(f"  ├─ TASA EXTRACCIÓN: {tasa_extraccion:>6.1f}%")
    print(f"  └─ 💧 DISPONIBLE:  {agua_disponible_promedio:>10.2f} ± {agua_disponible_std:>6.2f} hm³/año  {estado}")

# ============================================================================
# GUARDAR DATOS PARA LA PARTE 2
# ============================================================================

print("\n\n" + "="*150)
print("✅ PARTE 1 COMPLETADA - Guardando datos para PARTE 2...")
print("="*150)

import pickle

datos_para_parte2 = {
    'balance_por_acuifero': balance_por_acuifero,
    'resultados_finales': resultados_finales,
    'datos_conagua': datos_conagua,
    'precip_df': precip_df,
    'evap_df': evap_df,
    'grace_combinado': grace_combinado,
    'grace_anual': grace_anual,
    'OUTPUT_DIR': OUTPUT_DIR,
}

with open(f'{OUTPUT_DIR}/datos_parte1.pkl', 'wb') as f:
    pickle.dump(datos_para_parte2, f)

print(f"\n✓ Datos guardados en: {OUTPUT_DIR}/datos_parte1.pkl")
print(f"\n📌 Ahora ejecuta el script PARTE 2 para continuar con:")
print(f"   - PASO 6: Crear tabla comparativa")
print(f"   - PASO 7: Crear series temporales")
print(f"   - PASO 8: Visualizaciones")
print(f"   - PASO 9: Reporte final")
print(f"   - PASO 10: Guardar archivos")

print("\n" + "="*150 + "\n")


BALANCE HÍDRICO SUBTERRÁNEO CORREGIDO - PARTE 1 (PASOS 1-5)

[PASO 1] Cargando datos de precipitación y evaporación...
------------------------------------------------------------------------------------------------------------------------------------------------------
✓ Precipitación SMN cargada: 1188 registros, 64 estaciones
✓ Evaporación SMN cargada: 1056 registros, 57 estaciones
⚠ Advertencia: diferentes longitudes (Precip: 1188, Evap: 1056)
  → Sincronizadas a período común: 1056 registros

[PASO 2] Cargando datos de infiltración (GRACE)...
------------------------------------------------------------------------------------------------------------------------------------------------------
✓ GRACE mensual: 239 registros
✓ GRACE anual: 23 años
  Columnas GRACE: ['grace_tws_cm', 'grace_tws_change_cm', 'precipitation_mm', 'month']

[PASO 2.5] Sincronizando datos de precipitación y evaporación...
-----------------------------------------------------------------------------------------

In [6]:
# =====================================================================


BALANCE HÍDRICO SUBTERRÁNEO CORREGIDO - PARTE 2 (PASOS 6-10)

[CARGA] Cargando datos de PARTE 1...
✓ Datos cargados exitosamente
  - 5 acuíferos
  - 1056 registros temporales

[PASO 6] Creando tabla resumen...
------------------------------------------------------------------------------------------------------------------------------------------------------

                         Acuífero  Área (km²)  Recarga (hm³/yr)     Std Dev  Extracción (hm³/yr)  Tasa Extracción (%)  Agua Disponible (hm³/yr)     Std Dev
2101        Valle de Tecamachalco        3340         -8.224176   58.522912           217.756614                  NaN               -225.980790   58.522912
2102              Libres-Oriental        3995        -10.414805   69.012649           159.771638                  NaN               -170.186443   69.012649
2103  Atlixco-Izúcar de Matamoros        2658        -13.740660   98.960577           123.904179                  NaN               -137.644839   98.960577
2104         

ValueError: Length of values (239) does not match length of index (1056)